# Data Cleaning

In [34]:
import pandas as pd
import numpy as np
import re

## 1. Load the data

In [ ]:
data = pd.read_csv(
    'D:\\Estate-Miner\\Estate-Miner\\Scrapped_Data.csv',
    sep='\t',
    encoding='cp1252'
)
print(data.shape)
data.head()

(1852, 10)


,title,price,bedrooms,has_maid_room,bathrooms,area sqm,property_type,available from,location,amenities
0,A Full Sea view chalet 3BR for sale in north c...,16400000,3,Yes,3,1401507,Chalet,30-Apr-26,"June, Ras Al Hekma, North Coast","Unfurnished, Balcony, Built in Wardrobes, Cent..."
1,A Full Sea view chalet 3BR for sale in north c...,16400000,3,Yes,3,1401507,Chalet,30-Apr-26,"June, Ras Al Hekma, North Coast","Unfurnished, Balcony, Built in Wardrobes, Cent..."
2,Sea View Chalet 1BR In Fouka Bay Fully Finished,10500000,1,Yes,1,50538,Chalet,21-Jun-26,"Fouka Bay, Qesm Marsa Matrouh, North Coast","Unfurnished, Balcony, Built in Wardrobes, Cent..."
3,Sea View Chalet 1BR In Fouka Bay Fully Finished,10500000,1,Yes,1,50538,Chalet,21-Jun-26,"Fouka Bay, Qesm Marsa Matrouh, North Coast","Unfurnished, Balcony, Built in Wardrobes, Cent..."
4,chalet 3bed ready to move prime location old p...,17500000,3,Yes,3,1441550,Chalet,4-Jun-26,"Marassi, Sidi Abdel Rahman, North Coast","Unfurnished, Balcony, Built in Wardrobes, Cent..."


## 2. Initial inspection
Check dtypes, missing values, and a quick statistical summary before touching anything.

In [24]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1852 entries, 0 to 1851
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   title           1852 non-null   object
 1   price           1852 non-null   int64 
 2   bedrooms        1852 non-null   int64 
 3   has_maid_room   1852 non-null   object
 4   bathrooms       1852 non-null   int64 
 5   area sqm        1852 non-null   int64 
 6   property_type   1852 non-null   object
 7   available from  1852 non-null   object
 8   location        1852 non-null   object
 9   amenities       1852 non-null   object
dtypes: int64(4), object(6)
memory usage: 144.8+ KB


In [37]:
print("missing values per column:")
print(data.isna().sum())
print()
print("fully duplicated rows:", data.duplicated().sum())


missing values per column:
title             0
price             0
bedrooms          0
has_maid_room     0
bathrooms         0
area sqm          0
property_type     0
available from    0
location          0
amenities         0
dtype: int64

fully duplicated rows: 933


## 3. Standardize column names

In [38]:
data.columns = (
    data.columns
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)
data.columns.tolist()

['title',
 'price',
 'bedrooms',
 'has_maid_room',
 'bathrooms',
 'area_sqm',
 'property_type',
 'available_from',
 'location',
 'amenities']

## 4. Remove duplicate rows
The scrape captured many listings twice. Exact duplicate rows carry no extra information, so they are dropped.

In [39]:
before = len(data)
data = data.drop_duplicates().reset_index(drop=True)
after = len(data)
print(f"Removed {before - after} exact duplicate rows ({before} -> {after})")

Removed 933 exact duplicate rows (1852 -> 919)


## 5. Clean text columns

In [40]:
text_cols = data.select_dtypes(include="object").columns.tolist()
print("Text columns:", text_cols)

for col in text_cols:
    data[col] = (
        data[col]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

# Title-case the categorical-style columns; leave free-text title/amenities/location 
data['property_type'] = data['property_type'].str.strip().str.title()
data['has_maid_room'] = data['has_maid_room'].str.strip().str.title()

Text columns: ['title', 'has_maid_room', 'property_type', 'available_from', 'location', 'amenities']


## 6. Fix data types

In [41]:
data['has_maid_room'] = data['has_maid_room'].map({'Yes': True, 'No': False})

data['available_from'] = pd.to_datetime(data['available_from'], format='%d-%b-%y', errors='coerce')
print("Unparsed dates:", data['available_from'].isna().sum())

for col in ['price', 'bedrooms', 'bathrooms', 'area_sqm']:
    data[col] = pd.to_numeric(data[col], errors='coerce').astype('Int64')

data.dtypes

Unparsed dates: 0


title                     object
price                      Int64
bedrooms                   Int64
has_maid_room               bool
bathrooms                  Int64
area_sqm                   Int64
property_type             object
available_from    datetime64[ns]
location                  object
amenities                 object
dtype: object

## 7. Split `location` into structured parts

In [42]:
loc_parts = data['location'].str.split(',').apply(lambda parts: [p.strip() for p in parts])
data['region'] = loc_parts.apply(lambda parts: parts[-1])
data['sub_area'] = loc_parts.apply(lambda parts: parts[0])

data[['location', 'sub_area', 'region']].head()

,location,sub_area,region
0,"June, Ras Al Hekma, North Coast",June,North Coast
1,"Fouka Bay, Qesm Marsa Matrouh, North Coast",Fouka Bay,North Coast
2,"Marassi, Sidi Abdel Rahman, North Coast",Marassi,North Coast
3,"Zed East, 5th Settlement Compounds, The 5th Se...",Zed East,Cairo
4,"Silver Sands, Qesm Marsa Matrouh, North Coast",Silver Sands,North Coast


# 8. Splitting `amenities` into furnishing status + binary amenity flags
The `amenities` field mixes two unrelated concepts in one comma-separated string:
- a **furnishing status** (`Furnished` / `Partly furnished` / `Unfurnished`)
- a list of **actual physical amenities** (`Balcony`, `Private Pool`, `Security`, ...)

Mixing categorical attributes inside a free-text list column makes the field unusable for filtering, grouping, or modeling. The cells below separate the two concerns, clean the amenity text, and one-hot encode the amenities with `MultiLabelBinarizer` so each amenity becomes its own boolean feature.

## 8.1 Extract `furnishing_status`

In [ ]:
FURNISHING_LABELS = {
    "partly furnished": "Partly furnished",  
    "unfurnished": "Unfurnished",
    "furnished": "Furnished",
}

def extract_furnishing_status(amenities_str: str) -> str:
    """Return the furnishing status mentioned in an amenities string, or 'Unknown' if none is found."""
    text = amenities_str.lower()
    for needle, label in FURNISHING_LABELS.items():
        if needle in text:
            return label
    return "Unknown"

data['furnishing_status'] = data['amenities'].apply(extract_furnishing_status)

data['furnishing_status'].value_counts(dropna=False)

furnishing_status
Unfurnished         469
Partly furnished    320
Unknown              86
Furnished            44
Name: count, dtype: int64

## 8.2 Remove furnishing terms from the amenities text

In [46]:
FURNISHING_TOKENS = {label.lower() for label in FURNISHING_LABELS} | {label.lower() for label in FURNISHING_LABELS.values()}

def strip_furnishing(amenities_str: str) -> str:
    """Remove furnishing-status tokens from a comma-separated amenities string, keep everything else."""
    tokens = [t.strip() for t in amenities_str.split(',')]
    remaining = [t for t in tokens if t.lower() not in FURNISHING_TOKENS]
    return ', '.join(remaining)

data['amenities_no_furnishing'] = data['amenities'].apply(strip_furnishing)

data[['amenities', 'furnishing_status', 'amenities_no_furnishing']].head()

,amenities,furnishing_status,amenities_no_furnishing
0,"Unfurnished, Balcony, Built in Wardrobes, Cent...",Unfurnished,"Balcony, Built in Wardrobes, Central A/C, Cove..."
1,"Unfurnished, Balcony, Built in Wardrobes, Cent...",Unfurnished,"Balcony, Built in Wardrobes, Central A/C, Cove..."
2,"Unfurnished, Balcony, Built in Wardrobes, Cent...",Unfurnished,"Balcony, Built in Wardrobes, Central A/C, Cove..."
3,"Unfurnished, Balcony, Built in Wardrobes, Cent...",Unfurnished,"Balcony, Built in Wardrobes, Central A/C, Cove..."
4,"Partly furnished, Balcony, Built in Wardrobes,...",Partly furnished,"Balcony, Built in Wardrobes, Covered Parking, ..."


## 8.3 Split, clean, and deduplicate the amenity list per row

In [ ]:
def clean_amenities_list(amenities_str: str) -> list[str]:
    if not isinstance(amenities_str, str) or not amenities_str.strip():
        return []

    raw_tokens = amenities_str.split(',')

    cleaned_tokens = []
    seen = set()
    for token in raw_tokens:
        token = re.sub(r"\s+", " ", token).strip()  
        if not token:
            continue 
        
        normalized = token.title().replace("A/C", "A/C")

        if normalized.lower() not in seen:
            seen.add(normalized.lower())
            cleaned_tokens.append(normalized)

    return cleaned_tokens

data['amenities_clean_list'] = data['amenities_no_furnishing'].apply(clean_amenities_list)

data[['amenities_no_furnishing', 'amenities_clean_list']].head()

,amenities_no_furnishing,amenities_clean_list
0,"Balcony, Built in Wardrobes, Central A/C, Cove...","[Balcony, Built In Wardrobes, Central A/C, Cov..."
1,"Balcony, Built in Wardrobes, Central A/C, Cove...","[Balcony, Built In Wardrobes, Central A/C, Cov..."
2,"Balcony, Built in Wardrobes, Central A/C, Cove...","[Balcony, Built In Wardrobes, Central A/C, Cov..."
3,"Balcony, Built in Wardrobes, Central A/C, Cove...","[Balcony, Built In Wardrobes, Central A/C, Cov..."
4,"Balcony, Built in Wardrobes, Covered Parking, ...","[Balcony, Built In Wardrobes, Covered Parking,..."


## 8.4 One-hot encode amenities with `MultiLabelBinarizer`

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
amenity_binary_matrix = mlb.fit_transform(data['amenities_clean_list'])

amenity_cols = [f"amenity_{label.lower().replace(' ', '_').replace('/', '_')}" for label in mlb.classes_]

amenities_encoded = pd.DataFrame(amenity_binary_matrix, columns=amenity_cols, index=data.index)

print(f"Discovered {len(amenity_cols)} unique amenities")
amenities_encoded.head()

Discovered 18 unique amenities


,amenity_balcony,amenity_built_in_wardrobes,amenity_central_a_c,amenity_children's_pool,amenity_covered_parking,amenity_kitchen_appliances,amenity_lobby_in_building,amenity_maids_room,amenity_private_garden,amenity_private_pool,amenity_security,amenity_shared_gym,amenity_shared_pool,amenity_shared_spa,amenity_study,amenity_view_of_landmark,amenity_view_of_water,amenity_walk-in_closet
0,1,1,1,0,1,1,0,1,1,1,0,0,1,0,0,0,0,0
1,1,1,1,0,1,1,0,1,1,1,0,0,1,0,0,0,0,0
2,1,1,1,0,1,1,0,1,1,1,0,0,1,0,0,0,0,0
3,1,1,1,0,1,1,0,1,1,1,0,0,1,0,0,0,0,0
4,1,1,0,0,1,1,0,1,1,1,0,0,0,0,1,0,1,0


## 8.5 Assemble the final DataFrame

In [ ]:
df = pd.concat([data, amenities_encoded], axis=1)

assert (df[amenity_cols].sum(axis=1) == df['amenities_clean_list'].apply(len)).all(), \
    "Mismatch between binary amenity flags and cleaned amenity list"

print(df.shape)
df[['amenities', 'furnishing_status', 'amenities_clean_list'] + amenity_cols[:]].head()

(919, 33)


,amenities,furnishing_status,amenities_clean_list,amenity_balcony,amenity_built_in_wardrobes,amenity_central_a_c,amenity_children's_pool,amenity_covered_parking,amenity_kitchen_appliances,amenity_lobby_in_building,...,amenity_private_garden,amenity_private_pool,amenity_security,amenity_shared_gym,amenity_shared_pool,amenity_shared_spa,amenity_study,amenity_view_of_landmark,amenity_view_of_water,amenity_walk-in_closet
0,"Unfurnished, Balcony, Built in Wardrobes, Cent...",Unfurnished,"[Balcony, Built In Wardrobes, Central A/C, Cov...",1,1,1,0,1,1,0,...,1,1,0,0,1,0,0,0,0,0
1,"Unfurnished, Balcony, Built in Wardrobes, Cent...",Unfurnished,"[Balcony, Built In Wardrobes, Central A/C, Cov...",1,1,1,0,1,1,0,...,1,1,0,0,1,0,0,0,0,0
2,"Unfurnished, Balcony, Built in Wardrobes, Cent...",Unfurnished,"[Balcony, Built In Wardrobes, Central A/C, Cov...",1,1,1,0,1,1,0,...,1,1,0,0,1,0,0,0,0,0
3,"Unfurnished, Balcony, Built in Wardrobes, Cent...",Unfurnished,"[Balcony, Built In Wardrobes, Central A/C, Cov...",1,1,1,0,1,1,0,...,1,1,0,0,1,0,0,0,0,0
4,"Partly furnished, Balcony, Built in Wardrobes,...",Partly furnished,"[Balcony, Built In Wardrobes, Covered Parking,...",1,1,0,0,1,1,0,...,1,1,0,0,0,0,1,0,1,0


## 9. Save the cleaned data 

In [53]:
df.to_csv("./cleaned_data.csv", index=False)